# 03/03 — Mixed-effects model on signature scores

Per-spot signature scores (PIG, microglial activation, endocytosis) tested with random effects for mouse and section. One p-value per signature per region instead of thousands.

    score ~ treatment + region + treatment:region + (1|mouse) + (1|section)

Uses `statsmodels.MixedLM`. The relevant contrast is
`treatment[BRICHOS] - treatment[PBS]` per region.

In [ ]:
from __future__ import annotations
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

ROOT = Path.cwd().resolve()
while not (ROOT / 'utils').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('could not locate project root')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# regional-annotation h5ad lives on the processing volume
BASEDIR = Path('/Volumes/processing2/ST_BRICHOS/data')
H5AD_ORIENTED = BASEDIR / 'ST_BRICHOS_region_subcluster_oriented.h5ad'
H5AD_BASE     = BASEDIR / 'ST_BRICHOS_region_subcluster.h5ad'
H5AD = H5AD_ORIENTED if H5AD_ORIENTED.exists() else H5AD_BASE
COUNT_LAYER = 'counts'   # raw integer counts live here, not in .X

TBL = ROOT / 'results' / 'tables' / 'attenuation'
TBL.mkdir(parents=True, exist_ok=True)
FIG = ROOT / 'results' / 'figures' / 'manuscript'
FIG.mkdir(parents=True, exist_ok=True)

SAMPLE_KEY    = 'sample_id'           # change to 'library_id' if obs uses that
REGION_KEY    = 'anatomical_region'   # adjust to your obs column for regions
TREATMENT_KEY = 'treatment'
print('h5ad        :', H5AD)
print('count layer :', COUNT_LAYER)


In [ ]:
import statsmodels.formula.api as smf
from utils.attenuation import responder_zshift

adata = sc.read_h5ad(H5AD)
obs = adata.obs.copy()
obs['mouse']   = obs[SAMPLE_KEY]
obs['section'] = obs[SAMPLE_KEY]   # adjust if section != sample
obs['region']  = obs[REGION_KEY]
obs['treatment'] = obs[TREATMENT_KEY]


### Compute signature scores if not already present

Replace `PIG_GENES` etc. with the gene lists you actually use. `sc.tl.score_genes` adds a column to `adata.obs` and assumes **log-normalised data in `.X`**. Do *not* overwrite `.X` with counts before running this — only do that for pseudobulk.

In [ ]:
from utils.gene_sets import (
    PIG_CHEN2020, MICROGLIA_HOMEOSTATIC, ENDOCYTOSIS,
    DAM_KEREN_SHAUL2017, ARM_SALA_FRIGERIO2019,
    filter_to_var,
)

signatures = {
    'PIG':         PIG_CHEN2020,
    'microglia':   MICROGLIA_HOMEOSTATIC,
    'endocytosis': ENDOCYTOSIS,
    'DAM':         DAM_KEREN_SHAUL2017,
    'ARM':         ARM_SALA_FRIGERIO2019,
}
for name, genes in signatures.items():
    present = filter_to_var(genes, adata.var_names)
    if not present:
        print(f'!! {name}: no genes match var_names; skipping')
        continue
    sc.tl.score_genes(adata, present, score_name=f'{name}_score',
                      use_raw=False)
    obs[f'{name}_score'] = adata.obs[f'{name}_score'].values
    print(f'  {name}: scored {len(present)} / {len(genes)} genes')


### Mixed model per signature

In [ ]:
def fit_to_df(fit, signature):
    ci = fit.conf_int()
    ci.columns = ['ci_low', 'ci_high']
    out = pd.DataFrame({
        'coef'   : fit.params,
        'se'     : fit.bse,
        'z'      : fit.tvalues,
        'pvalue' : fit.pvalues,
    }).join(ci, how='left')
    out.index.name = 'term'
    out = out.reset_index()
    out.insert(0, 'signature', signature)
    return out

results = []
for sig in ['PIG_score', 'microglia_score', 'endocytosis_score',
            'DAM_score', 'ARM_score']:
    if sig not in obs.columns:
        continue
    df = obs[[sig, 'treatment', 'region', 'mouse']].dropna()
    df = df.rename(columns={sig: 'score'})
    md_ = smf.mixedlm('score ~ C(treatment, Treatment("WT"))'
                      ' * C(region)',
                      data=df, groups=df['mouse'])
    fit = md_.fit(method='lbfgs', reml=False, maxiter=200)
    results.append(fit_to_df(fit, sig))

if results:
    mixed_df = pd.concat(results, ignore_index=True)
    mixed_df.to_csv(TBL / 'signature_mixed_model.tsv',
                    sep='\t', index=False)
    mixed_df.head(20)


### Notes

* If convergence warnings appear, try `reml=True` and `method='powell'`.
* For section-level random effect, add `re_formula='1'` and a `vc_formula={'section': '0 + C(section)'}`. statsmodels can be fussy; if it fights you, drop section and keep only mouse.
* Confounders (slide batch, sex, age) go in the fixed-effect formula — `+ C(slide_batch) + sex + age`.